In [ ]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.constant import Interval
import polars as pl
from pathlib import Path
from datetime import datetime
from vnpy.alpha import Segment, AlphaDataset
import pandas as pd
import numpy as np
import lightgbm as lgb
from factor_define import (
    FACTOR_REGISTRY,
    FACTOR_NAMES
)
import optuna
from optuna.integration import LightGBMPruningCallback
import pickle
import gc
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# Cell 2: 路径配置和AlphaLab创建
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [ ]:
# ============================================================================
# Cell 3: 时间配置
# ============================================================================
# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime(2026, 3, 31)
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = datetime(2026, 3, 31)
interval2 = Interval.DAILY

# 训练跨度
train_start = datetime(2018, 1, 1)
train_end = datetime(2023, 12, 31)

# 验证跨度
valid_start = datetime(2024, 1, 1)
valid_end = datetime(2024, 12, 31)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

In [ ]:
# ============================================================================
# Cell 4: 加载数据集
# ============================================================================
DATASET_NAME = 'v1'
dataset: AlphaDataset = lab.load_dataset(DATASET_NAME)

In [ ]:
# ============================================================================
# Cell 5: 从 Dataset 提取 numpy 数据的工具函数
# ============================================================================

def extract_numpy_from_dataset(dataset, segment):
    """
    从 AlphaDataset 提取 numpy 数组给原生 LightGBM 使用

    Parameters
    ----------
    dataset : AlphaDataset
        VNPY 数据集
    segment : Segment
        数据段 (TRAIN, VALID, TEST)

    Returns
    -------
    X : np.ndarray
        特征矩阵
    y : np.ndarray
        标签向量
    df_meta : pl.DataFrame
        包含 datetime 和 vt_symbol 的元数据（用于后续构建信号）
    """
    # 获取学习数据（经过预处理的）
    df = dataset.fetch_learn(segment)

    # 元数据（用于后续生成信号）
    meta_cols = ['datetime', 'vt_symbol']
    df_meta = df.select(meta_cols)

    # 特征列：去掉 datetime, vt_symbol, label
    feature_cols = [c for c in df.columns if c not in ['datetime', 'vt_symbol', 'label']]

    # 转换为 numpy
    X = df.select(feature_cols).to_numpy()
    y = df['label'].to_numpy()

    # 获取日期编码用于分组
    date_codes = df['datetime'].to_numpy()

    # 计算分组大小（每天有多少个样本）
    unique_dates, group_sizes = np.unique(date_codes, return_counts=True)

    print(f'{segment.name}: X.shape={X.shape}, y.shape={y.shape}, de_meta.shape={df_meta.shape}')
    print(f'{segment.name}: 交易日数量 = {len(unique_dates)}, 平均每天样本数 = {group_sizes.mean():.1f}')
    return X, y, df_meta,date_codes, group_sizes

# 提取训练集和验证集数据
print('提取训练数据...')
X_train, y_train, meta_train, date_train, group_train = extract_numpy_from_dataset(dataset, Segment.TRAIN)

print('\n提取验证数据...')
X_valid, y_valid, meta_valid, date_valid, group_valid = extract_numpy_from_dataset(dataset, Segment.VALID)

print('\n提取测试数据...')
X_test, y_test, meta_test, date_test, group_test = extract_numpy_from_dataset(dataset, Segment.TEST)

# 释放 dataset 内存
print('\n释放 dataset 内存...')
del dataset
gc.collect()
print('✅ dataset 已释放')

In [ ]:
# ============================================================================
# Cell 6: 损失函数-预测值与标签IC的相反数
# ============================================================================
def group_ic_metric(preds, train_data):
    """
    按日期分组计算平均 IC

    参数:
    - preds: 模型预测值
    - train_data: lgb.Dataset 对象，需要预先设置 group 信息

    返回:
    - (metric_name, metric_value, is_higher_better)
    """
    labels = train_data.get_label()

    # 获取分组信息（每天有多少个股票）
    group_sizes = train_data.get_group()

    if group_sizes is None:
        # 如果没有分组信息，计算全局 IC
        ic, _ = spearmanr(preds, labels)
        return 'ic', ic, True

    # 按分组计算 IC
    start_idx = 0
    ics = []

    for size in group_sizes:
        end_idx = start_idx + size

        group_preds = preds[start_idx:end_idx]
        group_labels = labels[start_idx:end_idx]

        # 避免全相同值的情况
        if len(np.unique(group_preds)) > 1 and len(np.unique(group_labels)) > 1:
            try:
                ic, _ = spearmanr(group_preds, group_labels)
                if not np.isnan(ic):
                    ics.append(ic)
            except:
                pass

        start_idx = end_idx

    # 计算平均 IC
    mean_ic = np.mean(ics) if ics else 0
    ic_std = np.std(ics) if ics else 0
    ic_ir = mean_ic / (ic_std + 1e-8)  # IC Information Ratio

    # 可以返回多个指标
    # return [('ic_ir', ic_ir, True),('mean_ic', mean_ic, True)]
    return 'mean_ic', mean_ic, True

In [ ]:
# ============================================================================
# Cell 7: Optuna 超参数优化
# ============================================================================
print('\n开始 Optuna 超参数优化...')

def objective(trial):
    """Optuna 目标函数，返回验证集 mean_ic"""

    # 定义超参数搜索空间
    params = {
        'objective': 'regression',
        'metric': 'None',
        'boosting_type': 'gbdt',
        'device': 'gpu',                     # 如果有 GPU 可用

        # 核心树参数
        'num_leaves': trial.suggest_int('num_leaves', 512, 1024, step=64),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 200, step=10),

        # 学习参数
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),

        # 正则化
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),

        # 固定部分
        'verbose': -1,
        'seed': 42,
        'num_threads': -1,
    }

    # 创建数据集（每次 trial 都需重新创建，确保分组信息正确）
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
    train_data.set_group(group_train)
    valid_data.set_group(group_valid)

    # 训练参数（调参阶段可适当减少最大轮数）
    num_boost_round = 3000
    early_stopping_rounds = 200

    # 关键修正：明确指定验证集名称和指标名称
    valid_name = 'valid'
    metric_name = 'mean_ic'

    # 添加 Optuna 剪枝回调（可选）
    pruning_callback = LightGBMPruningCallback(
        trial,
        metric_name,           # 指标名称
        valid_name=valid_name  # 验证集名称（关键！）
    )

    # 训练模型
    model = lgb.train(
        params,
        train_data,
        num_boost_round=num_boost_round,
        valid_sets=[valid_data],
        valid_names=['valid'],
        feval=group_ic_metric,
        callbacks=[
            lgb.early_stopping(early_stopping_rounds),
            pruning_callback,                 # 启用剪枝
            # lgb.log_evaluation(period=100)   # 可适当减少输出频率
        ]
    )

    # 返回最佳验证 mean_ic
    best_score = model.best_score['valid']['mean_ic']
    return best_score


# 创建 Optuna Study
study = optuna.create_study(
    direction='maximize',
    study_name='lgbm_mf_optimization',
    storage=None,                            # 可改为 SQLite 路径持久化
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=30,
        n_warmup_steps=50,
        interval_steps=20
    ),
    sampler=optuna.samplers.TPESampler(seed=42),
)

# 执行优化（根据时间/算力调整 n_trials）
n_trials = 100
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print('\n===== Optuna 优化结果 =====')
print(f'最佳 trial: {study.best_trial.number}')
print(f'最佳验证 mean_ic: {study.best_value:.6f}')
print('最佳参数:')
for key, value in study.best_params.items():
    print(f'  {key}: {value}')



In [ ]:
# ============================================================================
# Cell 8: 使用最佳参数重新训练
# ============================================================================
print('\n使用最佳参数训练模型...')
best_params = study.best_params
final_params = {
    'objective': 'regression',
    'metric': 'None',
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'verbose': -1,
    'seed': 42,
    'num_threads': -1,
}
final_params.update(best_params)
# 训练参数
num_boost_round = 10000       # 最大迭代次数
early_stopping_rounds = 400   # 早停轮数
# 重新构建数据集
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
train_data.set_group(group_train)
valid_data.set_group(group_valid)

model = lgb.train(
    final_params,
    train_data,
    num_boost_round=num_boost_round,
    valid_sets=[valid_data],
    valid_names=['valid'],
    feval=group_ic_metric,
    callbacks=[
        lgb.early_stopping(early_stopping_rounds),
        lgb.log_evaluation(period=1)
    ]
)

print('\n训练完成!')
print(f'最佳迭代轮数: {model.best_iteration}')
print(f'最佳验证 : {model.best_score["valid"]["mean_ic"]:.6f}')

In [ ]:
# ============================================================================
# Cell 9: 特征重要性分析
# ============================================================================

print('\n特征重要性分析...')

# 获取特征重要性
importance = model.feature_importance(importance_type='gain')
feature_names = [f'{factor}_lag_{lag}' for factor in FACTOR_NAMES for lag in range(1, 11)]

# 创建重要性 DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print('Top 20 重要特征:')
print(importance_df.head(120))

In [ ]:
# ============================================================================
# Cell 10: 生成回测信号
# ============================================================================
print('\n在测试集上预测...')

# 预测
predictions = model.predict(X_test, num_iteration=model.best_iteration)

print(f'预测完成，预测样本数: {len(predictions)}')

# 构建信号 DataFrame
signal = meta_test.with_columns([
    pl.Series('signal', predictions)
])

print(f'\n信号数据形状: {signal.shape}')
print('信号数据预览:')
print(signal.head(10))

In [ ]:
# ============================================================================
# Cell 11: 保存模型和信号
# ============================================================================
MODEL_NAME = 'v1'
SIGNAL_NAME = 'v1'

# 保存 LightGBM 模型
MODEL_PICKLE_PATH = LAB_PATH / 'model' / f'{MODEL_NAME}.pkl'
with open(MODEL_PICKLE_PATH, 'wb') as f:
    pickle.dump({
        'model': model,
        'params': final_params,
        'best_iteration': model.best_iteration,
        'best_score': model.best_score
    }, f)
print(f'模型已保存: {MODEL_PICKLE_PATH}')

# 保存信号
SIGNAL_PARQUET_PATH = LAB_PATH / 'signal' / f'{SIGNAL_NAME}.parquet'
SIGNAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
signal.write_parquet(str(SIGNAL_PARQUET_PATH))
print(f'信号已保存: {SIGNAL_PARQUET_PATH}')

In [ ]:
with pd.option_context('display.max_rows', None):
    print(importance_df.head(120))